In [1]:
import numpy as np

def contrast_segmentation(img):

    # ---------- 1. Compute histogram ----------
    hist = np.bincount(img.ravel(), minlength=256)

    # ---------- 2. Otsu threshold ----------
    total_pixels = hist.sum()
    intensity = np.arange(256)

    total_mean = np.sum(intensity * hist)

    sumB = 0.0
    wB = 0.0
    max_var = 0.0
    optimal_threshold = 0

    for t in range(256):
        wB += hist[t]
        if wB == 0:
            continue

        wF = total_pixels - wB
        if wF == 0:
            break

        sumB += t * hist[t]

        mB = sumB / wB
        mF = (total_mean - sumB) / wF

        var_between = wB * wF * (mB - mF) ** 2

        if var_between > max_var:
            max_var = var_between
            optimal_threshold = t

    threshold = optimal_threshold

    # ---------- 3. Determine foreground side ----------
    above = img >= threshold
    below = img < threshold

    # smaller region = foreground
    if np.count_nonzero(above) < np.count_nonzero(below):
        fg_above = True
    else:
        fg_above = False

    return threshold, fg_above

In [2]:
import numpy as np


def generate_patch(img, conf, score_map, size, c, max_iter=1000):

    x, y = size

    flat_scores = score_map.ravel()

    if flat_scores.sum() == 0:
        return (-1, -1), 0, False

    # ---------- Pre-sort indices (descending order) ----------
    sorted_idx = np.argsort(flat_scores)[::-1]

    count_iter = 0

    for idx in sorted_idx:

        if count_iter >= max_iter:
            break

        if flat_scores[idx] == 0:
            break

        h, w = np.unravel_index(idx, score_map.shape)

        # ---------- Extract patches ----------
        img_patch = img[h:h+x, w:w+y]
        conf_patch = conf[h:h+x, w:w+y]

        # ---------- Segmentation ----------
        threshold, fg_above = contrast_segmentation(img_patch)

        # ---------- Foreground mask ----------
        if fg_above:
            mask = img_patch >= threshold
        else:
            mask = img_patch < threshold

        # ---------- Efficient validation ----------
        count = np.count_nonzero(mask)
        if count == 0:
            count_iter += 1
            continue

        mean_conf_fg = np.sum(conf_patch * mask) / count

        if mean_conf_fg > c:
            return (int(h), int(w)), float(threshold), bool(fg_above)

        count_iter += 1

    return (-1, -1), 0, False

In [3]:
import numpy as np
import random
from tqdm import tqdm


# ---------- Score map computation (using integral image) ----------

def patch_mean(integral, h, w, x, y):
    h2, w2 = h + x - 1, w + y - 1

    total = integral[h2, w2]
    if h > 0:
        total -= integral[h - 1, w2]
    if w > 0:
        total -= integral[h2, w - 1]
    if h > 0 and w > 0:
        total += integral[h - 1, w - 1]

    return total / (x * y)


def build_score_map(conf, size):
    x, y = size

    integral = conf.cumsum(axis=0).cumsum(axis=1)

    A = integral[x-1:, y-1:]
    B = np.pad(integral[:-x, y-1:], ((1,0),(0,0)))
    C = np.pad(integral[x-1:, :-y], ((0,0),(1,0)))
    D = np.pad(integral[:-x, :-y], ((1,0),(1,0)))

    score_map = (A - B - C + D) / (x * y)
    return score_map


# ---------- Update score map ----------
def update_score_map(score_map, h, w, x, y):
    score_map[h:h+x, w:w+y] = 0
    return score_map


# ---------- Main function ----------
import numpy as np
import random
from tqdm import tqdm


def patch_generater(
    target_patches,
    image_memmap_path,
    conf_memmap_path,
    image_shape,
    conf_shape,
    size,
    c,
    max_iter_per_image=1000,
    max_failures=100
):

    num_images = image_shape[0]
    x, y = size

    # ---------- Load full memmaps into RAM ----------
    images = np.asarray(np.memmap(image_memmap_path, dtype=np.uint8, mode='r', shape=image_shape))
    conf_maps = np.asarray(np.memmap(conf_memmap_path, dtype=np.float32, mode='r', shape=conf_shape))

    # ---------- Build score maps ----------
    score_maps = [
        build_score_map(conf_maps[i], size)
        for i in tqdm(range(num_images), desc="Building score maps")
    ]

    # ---------- Initialize result ----------
    result = {i: [] for i in range(num_images)}

    failures = 0
    total_collected = 0

    pbar = tqdm(total=target_patches, desc="Patch Generation")

    while total_collected < target_patches:

        # ---------- Random image ----------
        idx = random.randint(0, num_images - 1)

        img = images[idx]
        conf = conf_maps[idx]
        score_map = score_maps[idx]

        # ---------- Generate patch ----------
        (h, w), threshold, fg_above = generate_patch(
            img, conf, score_map, size, c, max_iter=max_iter_per_image
        )

        if h == -1:
            failures += 1
            pbar.set_postfix({"failures": failures})

            if failures >= max_failures:
                break
            continue

        # ---------- Success ----------
        failures = 0
        result[idx].append(((h, w), threshold, fg_above))
        total_collected += 1

        pbar.update(1)
        pbar.set_postfix({"failures": failures})

        # ---------- Update score map ----------
        score_maps[idx] = update_score_map(score_map, h, w, x, y)

    pbar.close()

    return result

In [4]:

# ---------- 1. Extract 49 image patches (optimized) ----------
def extract_image_patches(image_stack, h, w, x, y):
    """
    image_stack: (49, H, W) numpy array preferred
    Returns: (49, x, y)
    """
    return image_stack[:, h:h+x, w:w+y]


import numpy as np
from numba import njit


@njit
def compute_median_depth(img, depth_map, h, w, x, y, threshold, fg_above):

    vals = np.empty(x * y, dtype=np.uint8)
    count = 0

    for i in range(x):
        for j in range(y):

            img_val = img[h + i, w + j]

            if fg_above:
                cond = img_val >= threshold
            else:
                cond = img_val < threshold

            if cond:
                vals[count] = depth_map[h + i, w + j]
                count += 1

    if count == 0:
        return 0

    temp = vals[:count].copy()
    temp.sort()   # <-- FIX

    k = count // 2
    return temp[k]


In [5]:
import json
import numpy as np
from tqdm import tqdm


def build_dataset(
    result,
    image_memmap_path,
    depth_memmap_path,
    org_image_memmap_path,
    image_shape,
    depth_shape,
    org_image_shape,
    raw_to_org_ratio,
    size,
    patches_path,
    depth_out_path,
    org_patch_path,
    meta_path,
    flush_interval=1000
):

    x, y = size
    X = x*raw_to_org_ratio
    Y = y*raw_to_org_ratio

    total_patches = sum(len(v) for v in result.values())

    # ---------- Load input memmaps ----------
    image_memmap = np.memmap(image_memmap_path, dtype=np.uint8, mode='r', shape=image_shape)
    depth_memmap = np.memmap(depth_memmap_path, dtype=np.uint8, mode='r', shape=depth_shape)
    org_image_memmap = np.memmap(org_image_memmap_path, dtype=np.uint8, mode='r', shape=org_image_shape)

    # ---------- Create output memmaps ----------
    patches_array = np.memmap(patches_path, dtype=np.uint8, mode='w+', shape=(total_patches, 49, X, Y))
    focus_labels = np.memmap(depth_out_path, dtype=np.uint8, mode='w+', shape=(total_patches,))  # renamed
    org_patches = np.memmap(org_patch_path, dtype=np.uint8, mode='w+', shape=(total_patches, x, y))
    
    # ---------- Focal length array ----------
    slice_focal_length = np.array([
        3910.92,2289.27,1508.71,1185.83,935.91,801.09,700.37,605.39,546.23,
        486.87,447.99,407.40,379.91,350.41,329.95,307.54,291.72,274.13,261.53,
        247.35,237.08,225.41,216.88,207.10,198.18,191.60,183.96,178.29,171.69,
        165.57,160.99,155.61,150.59,146.81,142.35,138.98,134.99,131.23,127.69,
        124.99,121.77,118.73,116.40,113.63,110.99,108.47,106.54,104.23,102.01
    ], dtype=np.float32)

    idx_global = 0
    pbar = tqdm(total=total_patches, desc="Building dataset")

    for img_idx in result:

        image_stack = np.asarray(image_memmap[img_idx])
        org_img = np.asarray(org_image_memmap[img_idx])
        depth_map = np.asarray(depth_memmap[img_idx])

        for (h, w), threshold, fg_above in result[img_idx]:
            
            H = h*raw_to_org_ratio
            W = w*raw_to_org_ratio
            
            org_patches[idx_global] = org_img[h:h+x, w:w+y]
            patches_array[idx_global] = image_stack[:, H:H+X, W:W+Y]

            # ---------- Median depth ----------
            median_depth = compute_median_depth(
                org_img, depth_map, h, w, x, y, threshold, fg_above
            )
        
            # ---------- Predicted focus ----------
            approx = median_depth / 255.0

            max_val = 3.9
            min_val = 0.1

            metre = (max_val * min_val) / (max_val - (max_val - min_val) * approx)
            metre *= 1000.0

            # ---------- Closest index ----------
            focus_label = np.argmin(np.abs(slice_focal_length - metre))

            focus_labels[idx_global] = focus_label  # changed target

            idx_global += 1
            pbar.update(1)

            if idx_global % flush_interval == 0:
                patches_array.flush()
                focus_labels.flush()
                org_patches.flush()

    pbar.close()

    patches_array.flush()
    focus_labels.flush()
    org_patches.flush()
    
    metadata = {
        "patches": {"shape": (total_patches, 49, X, Y), "dtype": "uint8"},
        "focus_labels": {"shape": (total_patches,), "dtype": "uint8"},
        "org_patches": {"shape": (total_patches, x, y), "dtype": "uint8"}
    }
    
    with open(meta_path, "w") as f:
        json.dump(metadata, f)

    return patches_array, focus_labels, org_patches

In [6]:
import json

def load_metadata(meta_path):
    with open(meta_path, "r") as f:
        metadata = json.load(f)

    return metadata

In [7]:
dataset_type = "Test"

data_path = "/mnt/Velocity_Vault/Datasets/Autofocus/"+dataset_type
memmap_path = "/mnt/Velocity_Vault/Datasets/Autofocus/Memory/"+dataset_type

org_path = memmap_path +"/org_images.mm"
raw_path = memmap_path +"/raw_images.mm"
depth_path = memmap_path +"/depth_images.mm"
conf_path = memmap_path +"/conf_images.mm"

meta_path = memmap_path +"/meta_data.txt"
meta_data = load_metadata(meta_path)

target_patches = 1000
patch_size = (32,32)

In [8]:
patch_dict = patch_generater(
    target_patches,
    org_path,
    conf_path,
    meta_data['org']['shape'],
    meta_data['conf']['shape'],
    (32,32),
    0.7,
    max_iter_per_image=1000,
    max_failures=100
)

Patch Generation: 100%|██████████| 1000/1000 [00:08<00:00, 114.05it/s, failures=0]


In [9]:

dataset_path = "/mnt/Velocity_Vault/Datasets/Autofocus/Dataset/"+dataset_type

patch_path = dataset_path+"/patch.mm"
label_path = dataset_path+"/label.mm"
org_patch_path = dataset_path+"/org_patch.mm"

meta_data_path = dataset_path+"/meta.txt"

In [10]:

patch_dict, focus_labels, org_patches = build_dataset(
    patch_dict,
    raw_path,
    depth_path,
    org_path,
    meta_data['raw']['shape'],
    meta_data['depth']['shape'],
    meta_data['org']['shape'],
    4,
    (32,32),
    patch_path,
    label_path,
    org_patch_path,
    meta_data_path,
    flush_interval=1000
)

Building dataset: 100%|██████████| 1000/1000 [00:03<00:00, 332.72it/s]
